In [ ]:
# 노트북에서 루트기준으로 임포트 탐색할수있게 설정
from pathlib import Path
import sys

BASE_DIR = Path.cwd().parent.parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))


In [ ]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

from pilos.storage.comment_jsonl import load_comment_dataframe

# 기본 초기 품사 선택
TARGET_TAG_PREFIXES = (
    "NN",   # 일반명사, 고유명사, 의존명사
    "VV",   # 동사
    "VA",   # 형용사
    "MAG",  # 일반부사: 안, 못 등 포함
    "SL",   # 영어
    "SN",   # 숫자
)
# 조금더 빡센 품사선택 명사중심
TARGET_TAG_PREFIXES = (
    "NN",
    "SL",
    "SN",
)
# 조금 덜 빡센 품사선택 명사+동사중심
TARGET_TAG_PREFIXES = (
    "NN",
    "VV",
    "VA",
    "SL",
    "SN"
)
DATA_DIR = BASE_DIR/"data"
DATA_PATH = DATA_DIR/"processed"/"tokenized_comments.jsonl"


# 판다스 디스플레이 설정
pd.set_option("display.max_colwidth", None)

In [ ]:
# 토큰화된 데이터 가져오기
comments_df = load_comment_dataframe(
    input_path=DATA_PATH
    )


In [ ]:
# 백터화에 필요한 컬럼 선택하기
vectorization_df = comments_df[
    [
        "comment_id",
        "stock_code",
        "created_at",
        "kiwi_tokens",
        "text",
    ]
].copy()



In [ ]:
# TF_IDF 백터나이저에 입력할수있는 문자열 만들기
def build_tfidf_text(tokens: list[dict]) -> str:
    """허용된 품사의 형태소만 공백으로 연결한다."""
    selected_forms = []

    for token in tokens:
        form = token["form"]
        tag = token["tag"]

        if tag.startswith(TARGET_TAG_PREFIXES):
            selected_forms.append(form)

    return " ".join(selected_forms)


In [ ]:
sample_df = vectorization_df.head(5).copy()

sample_df["tfidf_text"] = (
    sample_df["kiwi_tokens"]
    .apply(build_tfidf_text)
)
sample_df["tfidf_text"].column

In [ ]:
# 전처리된 데이터에서 백터입력 텍스트 생성
vectorization_df["tfidf_text"] = (
    vectorization_df["kiwi_tokens"]
    .apply(build_tfidf_text)
)

In [ ]:
# 허용품사가 아닌 품사들로만 모여서 빈문자열이 된 컬럼 추적 및 제거
empty_mask = vectorization_df["tfidf_text"].str.strip().eq("")

print("전체 댓글 수:", len(vectorization_df))
print("빈 TF-IDF 문서 수:", empty_mask.sum())
tfidf_df = (
    vectorization_df.loc[~empty_mask]
    .reset_index(drop=True)
    .copy()
)

print("실제 벡터화 문서 수:", len(tfidf_df))

In [ ]:
tfidf_df["date"] = pd.to_datetime(
    tfidf_df["created_at"]
).dt.date



| 옵션 | 지금 설정 | 이유 |
|---|---|---|
| `tokenizer` | `str.split` | Kiwi 토큰을 공백 기준으로 사용 |
| `token_pattern` | `None` | 사이킷런의 재토큰화 방지 |
| `lowercase` | `False` | 기존 토큰 형태 유지 |
| `min_df` | `2` | 1회성 오타·희귀 표현 일부 제거 |
| `max_df` | 기본값 | 먼저 흔한 단어 확인 |
| `max_features` | 기본값 | 전체 단어 수 확인 |
| `ngram_range` | `(1, 1)` | 우선 단일 단어부터 확인 |
| `stop_words` | 없음 | 결과를 보고 선정 |
| `sublinear_tf` | `False` | 우선 기본 계산 확인 |

In [ ]:
# 백터라이저 소환
vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False,
    min_df=2,
)

In [ ]:
tfidf_matrix = vectorizer.fit_transform(
    tfidf_df["tfidf_text"]
)

In [ ]:
print("행렬 크기:", tfidf_matrix.shape)
print("0이 아닌 값 개수:", tfidf_matrix.nnz)
print("자료형:", tfidf_matrix.dtype)

In [ ]:
row_index = 0
document = tfidf_df.iloc[row_index]
feature_names = vectorizer.get_feature_names_out()
# 원문과 TF-IDF 입력문
display(
    tfidf_df.loc[
        [tfidf_df.index[row_index]],
        ["text", "tfidf_text"],
    ].T
)

# Kiwi 토큰
display(
    pd.DataFrame(document["kiwi_tokens"])[["form", "tag"]]
)

# TF-IDF 결과
row = tfidf_matrix.getrow(row_index)

display(
    pd.DataFrame({
        "word": feature_names[row.indices],
        "tfidf": row.data,
    }).sort_values("tfidf", ascending=False)
)

In [ ]:
# 백터값을 일별로 집계하여 일별 단어 흐름보기
group_indices = tfidf_df.groupby(
    ["stock_code", "date"]
).indices
group_key = list(group_indices.keys())[0]
row_indices = group_indices[group_key]

daily_mean_vector = tfidf_matrix[row_indices].mean(axis=0)
daily_mean_values = daily_mean_vector.A1

daily_top_words = pd.DataFrame({
    "word": feature_names,
    "mean_tfidf": daily_mean_values,
}).sort_values(
    "mean_tfidf",
    ascending=False,
)

In [ ]:
for key, indices in list(group_indices.items())[:2]:
    print(key)
    print("댓글 수:", len(indices))

    vector = tfidf_matrix[indices].mean(axis=0).A1

    result = pd.DataFrame({
        "word": feature_names,
        "score": vector
    })

    print(
        result.sort_values(
            "score",
            ascending=False
        ).head(10)
    )

In [ ]:
print(group_indices)

print(list(group_indices.keys())[0])

print(group_indices[list(group_indices.keys())[0]])

In [ ]:
import numpy as np

rng = np.random.default_rng(43)

sample_rows = rng.choice(
    len(tfidf_df),
    size=15,
    replace=False,
)

sample_rows

In [ ]:
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", None)

feature_names = vectorizer.get_feature_names_out()

for row_index in sample_rows:
    document = tfidf_df.iloc[row_index]
    row = tfidf_matrix.getrow(row_index)

    display(
        Markdown(
            f"## 행 번호: {row_index} / 댓글 ID: {document['comment_id']}"
        )
    )

    # 원문과 품사 필터링 결과
    display(
        document[
            ["text", "tfidf_text"]
        ].rename(
            {
                "text": "원문",
                "tfidf_text": "TF-IDF 입력문",
            }
        ).to_frame("내용")
    )

    # 해당 댓글의 TF-IDF 상위 단어
    tfidf_result = pd.DataFrame({
        "word": feature_names[row.indices],
        "tfidf": row.data,
    }).sort_values(
        "tfidf",
        ascending=False,
    )

    display(tfidf_result.head(10))